# NHTSA Recall Lag Analysis — Data Cleaning Pipeline

## Project overview
This notebook loads, cleans, and standardizes NHTSA complaints and recall data so the two sources can be joined for manufacturer recall-response-time analysis.

Pipeline:
1. Load the raw complaints and recall flat files.
2. Assign source schemas and retain only columns needed for lag analysis.
3. Validate and standardize the dates used for the analysis.
4. Match manufacturer names to a canonical passenger-vehicle OEM list.
5. Parse component hierarchies into a shared vocabulary.
6. Prepare cleaned tables for export to SQL Server, where lag calculations will be performed.


## 1. Load complaints data

NHTSA ships complaints as a tab-delimited flat file with no header row, so  
columns are loaded positionally and named manually against the official  
NHTSA field dictionary (CMPL.txt) in the next step.

In [1]:
import pandas as pd
import csv

# Load the complaints data file
# - sep="\t": tab-separated format
# - header=None: no header row; we'll assign column names manually
# - quoting=csv.QUOTE_NONE: raw text, no quote handling
# - low_memory=False: load all data at once (not in chunks)
df = pd.read_csv(
    "../datasets/FLAT_CMPL.txt",
    sep="\t",
    header=None,
    quoting=csv.QUOTE_NONE,
    low_memory=False
)

# Report the raw row/column count to confirm the source loaded completely.
print(df.shape)

(2234970, 51)


In [2]:
# Inspect the first row to understand column mapping
# This helps us verify which column index contains what data

for i, value in enumerate(df.iloc[0]):    print(f"Column {i}: {value}")

Column 0: 1
Column 1: 958241
Column 2: Volvo Car USA, LLC
Column 3: VOLVO
Column 4: 760
Column 5: 1987.0
Column 6: N
Column 7: nan
Column 8: N
Column 9: 0
Column 10: 0
Column 11: ENGINE AND ENGINE COOLING:COOLING SYSTEM:RADIATOR ASSEMBLY
Column 12: EL CAJON    
Column 13: CA
Column 14: nan
Column 15: 19950103
Column 16: 19950103
Column 17: nan
Column 18: nan
Column 19: RADIATOR FAILED @ HIGHWAY SPEED OBSTRUCTING DRIVERS VISION TEMPORARY. PLEASE DESCRIBE DETAILS.  TT
Column 20: EVOQ
Column 21: N
Column 22: nan
Column 23: N
Column 24: N
Column 25: N
Column 26: nan
Column 27: nan
Column 28: nan
Column 29: nan
Column 30: nan
Column 31: nan
Column 32: nan
Column 33: nan
Column 34: nan
Column 35: nan
Column 36: nan
Column 37: nan
Column 38: nan
Column 39: nan
Column 40: nan
Column 41: nan
Column 42: nan
Column 43: nan
Column 44: nan
Column 45: V
Column 46: nan
Column 47: N
Column 48: N
Column 49: nan
Column 50: nan


## 2. Load recall data (pre/post-2010 files)

NHTSA splits recall history across two files — `FLAT_RCL_PRE_2010.txt` and  
`FLAT_RCL_POST_2010.txt` — purely as a storage/export artifact, not a schema  
change. Both are loaded separately here and combined into one table once  
columns are assigned.

In [3]:
# Define column names based on NHTSA complaints schema
# Each column maps to a specific field in the complaints data
columns = [
    "CMPLID",
    "ODINO",
    "MFR_NAME",
    "MAKETXT",
    "MODELTXT",
    "YEARTXT",
    "CRASH",
    "FAILDATE",
    "FIRE",
    "INJURED",
    "DEATHS",
    "COMPDESC",
    "CITY",
    "STATE",
    "VIN",
    "DATEA",
    "LDATE",
    "MILES",
    "OCCURENCES",
    "CDESCR",
    "CMPL_TYPE",
    "POLICE_RPT_YN",
    "PURCH_DT",
    "ORIG_OWNER_YN",
    "ANTI_BRAKES_YN",
    "CRUISE_CONT_YN",
    "NUM_CYLS",
    "DRIVE_TRAIN",
    "FUEL_SYS",
    "FUEL_TYPE",
    "TRANS_TYPE",
    "VEH_SPEED",
    "DOT",
    "TIRE_SIZE",
    "LOC_OF_TIRE",
    "TIRE_FAIL_TYPE",
    "ORIG_EQUIP_YN",
    "MANUF_DT",
    "SEAT_TYPE",
    "RESTRAINT_TYPE",
    "DEALER_NAME",
    "DEALER_TEL",
    "DEALER_CITY",
    "DEALER_STATE",
    "DEALER_ZIP",
    "PROD_TYPE",
    "REPAIRED_YN",
    "MEDICAL_ATTN",
    "VEHICLES_TOWED_YN",
    "STATE_OF_INCIDENT",
    "VEHICLE_OPERATOR"
]

df.columns = columns

In [4]:
# Preview representative records to verify that the assigned schema aligns with the raw values.
print(df.head())

   CMPLID   ODINO                MFR_NAME MAKETXT     MODELTXT  YEARTXT CRASH  \
0       1  958241      Volvo Car USA, LLC   VOLVO          760   1987.0     N   
1       2  958130      Ford Motor Company    FORD  THUNDERBIRD   1992.0     N   
2       3  958132       Kia America, Inc.     KIA       SEPHIA   1994.0     Y   
3       4  958133  Chrysler (FCA US, LLC)   DODGE          600   1987.0     N   
4       5  958137  Chrysler (FCA US, LLC)   DODGE      CARAVAN   1991.0     N   

     FAILDATE FIRE  INJURED  ...  DEALER_TEL DEALER_CITY DEALER_STATE  \
0         NaN    N        0  ...         NaN         NaN          NaN   
1  19941222.0    N        0  ...         NaN         NaN          NaN   
2  19941230.0    N        0  ...         NaN         NaN          NaN   
3  19941231.0    N        0  ...         NaN         NaN          NaN   
4  19941218.0    N        0  ...         NaN         NaN          NaN   

  DEALER_ZIP PROD_TYPE  REPAIRED_YN  MEDICAL_ATTN  VEHICLES_TOWED_YN  \
0 

In [5]:
import pandas as pd
import csv

# Load post-2010 recalls (Jan 1, 2010 onwards).
# NHTSA provides recalls in separate pre/post-2010 files; this is a source-storage split,
# not a schema difference, so the files can later be combined after validation.
rcl_post = pd.read_csv(
    "../datasets/FLAT_RCL_POST_2010.txt",
    sep="\t",
    header=None,
    quoting=csv.QUOTE_NONE,
    low_memory=False
)

print(rcl_post.shape)
print(rcl_post)

(244390, 29)
            0          1              2         3     4           5   \
0        81716  10V598000          ALTEC    AT35-G  1998     CSN 535   
1        81717  10V317000         PIERCE     IMPEL  2007  39-0139830   
2        81718  10V317000         PIERCE  VELOCITY  2007  39-0139830   
3        81719  10V317000         PIERCE   QUANTUM  2006  39-0139830   
4        81720  10V317000         PIERCE  VELOCITY  2008  39-0139830   
...        ...        ...            ...       ...   ...         ...   
244385  326101  26V481000  MERCEDES-BENZ   GLB 250  2022         NaN   
244386  326102  26V481000  MERCEDES-BENZ   GLB 250  2022         NaN   
244387  326103  26V481000  MERCEDES-BENZ   GLB 250  2022         NaN   
244388  326104  26V481000  MERCEDES-BENZ   GLB 250  2022         NaN   
244389  326105  26V481000  MERCEDES-BENZ   GLB 250  2021         NaN   

                                                       6   \
0                                               EQUIPMENT   


In [6]:
# Load pre-2010 recalls (before Jan 1, 2010) to preserve the historical records
# required when comparing complaint activity with later recall announcements.
rcl_pre = pd.read_csv(
    "../datasets/FLAT_RCL_PRE_2010.txt",
    sep="\t",
    header=None,
    quoting=csv.QUOTE_NONE,
    low_memory=False
)

print(rcl_pre.shape)

(81710, 29)


### Assign recall column names

Same schema applies to both pre- and post-2010 files, so one column list is  
reused for both before they're concatenated.

In [ ]:
# Define column names based on NHTSA recalls schema
# Same columns for both pre and post-2010 data; naming them makes the flat-file
# positions usable in later filtering, cleaning, and joining steps.
rcl_columns = [
    "RECORD_ID",
    "CAMPNO",
    "MAKETXT",
    "MODELTXT",
    "YEARTXT",
    "MFGCAMPNO",
    "COMPNAME",
    "MFGNAME",
    "BGMAN",
    "ENDMAN",
    "RCLTYPECD",
    "POTAFF",
    "ODATE",
    "INFLUENCED_BY",
    "MFGTXT",
    "RCDATE",
    "DATEA",
    "RPNO",
    "FMVSS",
    "DESC_DEFECT",
    "CONEQUENCE_DEFECT",
    "CORRECTIVE_ACTION",
    "NOTES",
    "RCL_CMPT_ID",
    "MFR_COMP_NAME",
    "MFR_COMP_DESC",
    "MFR_COMP_PTNO",
    "DO_NOT_DRIVE",
    "PARK_OUTSIDE"
]
# Assign column names to both pre and post-2010 recalls

rcl_post.columns = rcl_columns
rcl_pre.columns = rcl_columns

In [8]:
# Verify column names are consistently assigned
print("Pre-2010 columns:", rcl_pre.columns.tolist())
print("Post-2010 columns:", rcl_post.columns.tolist())

Pre-2010 columns: ['RECORD_ID', 'CAMPNO', 'MAKETXT', 'MODELTXT', 'YEARTXT', 'MFGCAMPNO', 'COMPNAME', 'MFGNAME', 'BGMAN', 'ENDMAN', 'RCLTYPECD', 'POTAFF', 'ODATE', 'INFLUENCED_BY', 'MFGTXT', 'RCDATE', 'DATEA', 'RPNO', 'FMVSS', 'DESC_DEFECT', 'CONEQUENCE_DEFECT', 'CORRECTIVE_ACTION', 'NOTES', 'RCL_CMPT_ID', 'MFR_COMP_NAME', 'MFR_COMP_DESC', 'MFR_COMP_PTNO', 'DO_NOT_DRIVE', 'PARK_OUTSIDE']
Post-2010 columns: ['RECORD_ID', 'CAMPNO', 'MAKETXT', 'MODELTXT', 'YEARTXT', 'MFGCAMPNO', 'COMPNAME', 'MFGNAME', 'BGMAN', 'ENDMAN', 'RCLTYPECD', 'POTAFF', 'ODATE', 'INFLUENCED_BY', 'MFGTXT', 'RCDATE', 'DATEA', 'RPNO', 'FMVSS', 'DESC_DEFECT', 'CONEQUENCE_DEFECT', 'CORRECTIVE_ACTION', 'NOTES', 'RCL_CMPT_ID', 'MFR_COMP_NAME', 'MFR_COMP_DESC', 'MFR_COMP_PTNO', 'DO_NOT_DRIVE', 'PARK_OUTSIDE']


In [9]:
# Combine pre-2010 and post-2010 recalls into a single table.
# The source split is only a file-storage artifact, so direct concatenation retains
# the full recall history in one analysis-ready dataset.
# ignore_index=True renumbers rows sequentially
recalls = pd.concat(
    [rcl_pre, rcl_post],
    ignore_index=True
)

print(f"Combined recalls shape: {recalls.shape}")

Combined recalls shape: (326100, 29)


In [10]:
# Select only the columns needed for lag analysis from complaints and recalls.
# Keeping the focused subset reduces memory use before the heavier cleaning steps
# while retaining identifiers, OEM/component fields, dates, and defect context.
complaints_cols = ["CMPLID", "ODINO", "MFR_NAME", "MAKETXT", "MODELTXT", "YEARTXT",
                    "COMPDESC",'FAILDATE', "DATEA", "CRASH", "FIRE", "INJURED", "DEATHS", "CDESCR"]

recalls_cols = ["CAMPNO", "MFGNAME", "MAKETXT", "MODELTXT", "YEARTXT",
                "COMPNAME", "RCDATE",'ODATE', "DESC_DEFECT", "CONEQUENCE_DEFECT"]

complaints_df = df[complaints_cols]
recalls_df = recalls[recalls_cols]

In [ ]:
# Check for null/missing values in each date column
# This helps us decide which dates to keep vs. drop
print("=== Complaints Date Quality ===")
for col in ["FAILDATE", "DATEA"]:
    n_null = complaints_df[col].isna().sum()
    pct_null = n_null / len(complaints_df) * 100
    print(f"{col}: {n_null} nulls ({pct_null:.2f}%)")

print("\n=== Recalls Date Quality ===")
for col in ["RCDATE", "ODATE"]:
    n_null = recalls_df[col].isna().sum()
    pct_null = n_null / len(recalls_df) * 100
    print(f"{col}: {n_null} nulls ({pct_null:.2f}%)")

=== Complaints Date Quality ===
FAILDATE: 110144 nulls (4.93%)
DATEA: 0 nulls (0.00%)

=== Recalls Date Quality ===
ODATE: 13875 nulls (4.25%)


In [12]:
# Detect suspicious or malformed date values
date_cols_complaints = ["FAILDATE", "DATEA"]
date_cols_recalls = ["RCDATE", "ODATE"]

def check_bad_dates(df, cols):
    """Check for bad date formats: 00000000 (placeholder), wrong length, or string 'nan'"""
    for col in cols:
        vals = df[col].astype(str)
        # Find dates that are 00000000 (NHTSA placeholder), not 8 chars (malformed), or literal string 'nan'
        bad = vals[(vals == "00000000") | (vals.str.len() != 8) | (vals == "nan")]
        print(f"{col}: {len(bad)} suspicious values out of {len(df)}")
        print(vals.value_counts().head(3))  # peek at most common values
        print("---")

check_bad_dates(recalls_df, date_cols_recalls)
check_bad_dates(complaints_df, date_cols_complaints)

RCDATE: 0 suspicious values out of 326100
RCDATE
20241209    21248
20210205    19301
20220520    13826
Name: count, dtype: int64
---
ODATE: 326100 suspicious values out of 326100
ODATE
20250127.0    19322
20210312.0    18739
nan           13875
Name: count, dtype: int64
---
FAILDATE: 2234969 suspicious values out of 2234970
FAILDATE
nan           110144
19010101.0      5647
20160601.0      2885
Name: count, dtype: int64
---
DATEA: 0 suspicious values out of 2234970
DATEA
20040422    5231
20020514    4329
19990217    2259
Name: count, dtype: int64
---


## 3. Date cleaning

Each source has two date fields, but the lag analysis uses one reliable date 
per table. `DATEA` is retained for complaints because it records when NHTSA 
logged the complaint and has complete coverage; `FAILDATE` is self-reported 
and can be missing or unreliable. `RCDATE` is retained for recalls because it 
represents the public recall date; `ODATE` is an internal processing field.

The next cells audit nulls and placeholder values, remove the unused fields, 
convert the retained fields to datetimes, and validate their ranges before 
any lag calculation.

In [13]:
# Drop unreliable date columns
# - FAILDATE (complaints): too sparse and unreliable; use DATEA instead
# - ODATE (recalls): internal processing date; use RCDATE (public announcement) instead

recalls_df = recalls_df.drop(columns=["ODATE"])
complaints_df = complaints_df.drop(columns=["FAILDATE"])
print("Dropped ODATE from recalls and FAILDATE from complaints")

Dropped ODATE from recalls and FAILDATE from complaints


In [14]:
# Convert DATEA to real datetime.
# It's currently stored as int/float (e.g. 20040422) not a date pandas understands,
# so string ops (like filtering by month) won't work until this is a proper datetime.
complaints_df["DATEA"] = pd.to_datetime(
    complaints_df["DATEA"].astype(str),   # cast to string first since format="%Y%m%d" needs text input
    format="%Y%m%d",
    errors="coerce"   # any value that doesn't match YYYYMMDD becomes NaT instead of crashing the whole conversion
)

# Same logic for RCDATE on the recalls side — this is your "official recall happened" date,
# needs to be a real datetime to later compute (recall_date - spike_date) as an actual time delta.
recalls_df["RCDATE"] = pd.to_datetime(
    recalls_df["RCDATE"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

# Sanity check: verify conversion worked and date ranges look realistic
# Expected ranges:
#   - Complaints: ~1995 to ~2023 (when this data was extracted)
#   - Recalls: ~1966 to ~2023 (NHTSA has historical recall data)

# If you see future dates or weird values, that's a red flagprint("Recalls RCDATE range:", recalls_df["RCDATE"].min(), "→", recalls_df["RCDATE"].max())

print("\n=== Date Range Validation ===")
print("Complaints DATEA range:", complaints_df["DATEA"].min(), "→", complaints_df["DATEA"].max())


=== Date Range Validation ===
Complaints DATEA range: 1995-01-03 00:00:00 → 2026-08-12 00:00:00


## 4. Manufacturer name standardization & OEM matching

Raw manufacturer names vary in casing, whitespace, and legal-entity suffixes  
(e.g. `Ford Motor Company` vs `FORD MTR CO`), and the field also includes  
non-OEM entities NHTSA regulates alongside cars (tire makers, child seat  
brands, trailer manufacturers). This section:
- standardizes casing/whitespace,
- matches raw names to a canonical passenger-vehicle OEM list using a manual  
  alias dictionary first, then RapidFuzz `token_set_ratio` as a fallback,
- filters both tables down to only rows that matched a real OEM.

In [15]:
# Standardize casing + whitespace on manufacturer/make columns.
# Raw NHTSA text has inconsistent spacing/casing (" chevrolet", "CHEVROLET ", "Chevrolet")
# which would otherwise create false-duplicate manufacturers when grouping/joining later.
for col in ["MFR_NAME", "MAKETXT"]:
    complaints_df[col] = complaints_df[col].str.strip().str.upper()

for col in ["MFGNAME", "MAKETXT"]:
    recalls_df[col] = recalls_df[col].str.strip().str.upper()

# Peek at distinct manufacturer names on both sides —
# this is how you'll spot mismatches like "CHEVROLET" vs "CHEVY" or "GM" vs "GENERAL MOTORS"
# before building the alias/crosswalk table.
print(complaints_df["MFR_NAME"].nunique(), "unique complaint manufacturers")
print(recalls_df["MFGNAME"].nunique(), "unique recall manufacturers")


1763 unique complaint manufacturers
3577 unique recall manufacturers


In [16]:

print(sorted(complaints_df["MFR_NAME"].dropna().unique()))   # sample
print(sorted(recalls_df["MFGNAME"].dropna().unique()))       # sample

['3M COMPANY TRAFFIC SAFETY SYSTEMS DIV.', '4-STAR TRAILERS, INC.', 'A&J VANS INC', 'A&M SYSTEMS, INC.', 'A-1 AUTO ELECTRIC', 'A.L. SOLUTIONS US', 'A2ZEV', 'AAI MOTORSPORTS USA, LLC', 'ABADDON PRODUCTS CO.', 'ABC ENGINEERING, CO', 'ABS REMORQUES', 'ACCESSORIES MARKETING, INC.', 'ACCESSORY DIST., INC.', 'ACCUBUILT, INC.', 'ACCURIDE CORPORATION', 'ACD TRIDON INC', 'ACE ELECTRIC CO., INC.', 'ACHIEVA RUBBER CORP.', 'ACRO TRAILER COMPANY', 'ACTION AUTO BODY, INC.', 'ACTIVE TECHNOLOGY', 'ACTRON', 'ADCO PRODUCTS, INC.', 'ADIENT, PLC', 'ADRIAN STEEL COMPANY', 'ADV BRK SYS', 'ADVANCED BUS INDUSTRIES, LLC', 'ADVANCED CARBON COMPOSITES INC.', 'ADVANCED ENGINE MANAGEMENT', 'ADVANCED FUEL COMPONENTS, INC', 'ADVENTIVE IDEAS, LLC', 'ADVENTURE MANUFACTURING', 'AERO COACH, INC.', 'AERO MANUFACTURING, INC.', 'AFCO ELECTRONICS', 'AFFINIA GROUP, INC.', 'AFX NORTH AMERICA, INC.', 'AGILITY FUEL SOLUTIONS', 'AGWAY TIRE CO.', 'AHNAFIELD', 'AIDIA, INC.', 'AIMCO PRODUCTS INC.', 'AIR-O-MATIC PWR STRNG CO.', 'AIR

In [17]:
import pandas as pd

# Canonical list of passenger car/light-truck OEMs.
# This is your "ground truth" — anything that doesn't match one of these
# (helmet makers, trailer companies, tire brands, etc.) gets filtered out.
oem_reference = [
    "FORD", "TOYOTA", "HONDA", "GENERAL MOTORS", "CHEVROLET", "GMC",
     "CHRYSLER", "STELLANTIS", "NISSAN",
    "HYUNDAI", "KIA", "SUBARU", "MAZDA", "VOLKSWAGEN", "PORSCHE",
    "BMW", "MERCEDES-BENZ", "TESLA", "VOLVO", "LAND ROVER", "JAGUAR",
    "MITSUBISHI", "INFINITI", "LINCOLN", "MINI",
    "FIAT", "ALFA ROMEO", "GENESIS", "RIVIAN", "LUCID", "POLESTAR",
    "SUZUKI", "SAAB", "SATURN", "PONTIAC", "OLDSMOBILE", "MERCURY",
    "ISUZU", "SCION", "SMART", "MASERATI", "FERRARI", "LAMBORGHINI",
    "BENTLEY", "ROLLS-ROYCE", "ASTON MARTIN", "MCLAREN"
]

print(len(oem_reference), "reference OEMs")

47 reference OEMs


In [18]:
import re
import pandas as pd
from rapidfuzz import process, fuzz

# --- Strip punctuation before matching (fixes "TESLA, INC." style misses) ---
def clean_name(name):
    if pd.isna(name):
        return name
    return re.sub(r"[^\w\s-]", "", name)

# --- Manual alias dictionary ---
manual_aliases = {
    "GENERAL MOTORS": ["GENERAL MOTORS LLC", "GENERAL MOTORS CORP", "GENERAL MOTORS COMPANY"],
    "FORD": ["FORD MOTOR COMPANY", "FORD MOTOR COMPANY TEST ADW AS", "FORD MTR CO-OVERSEAS"],
    "TOYOTA": ["TOYOTA MOTOR CORPORATION", "TOYOTA MOTOR CO LTD", "TOYOTA MOTOR NORTH AMERICA INC",
               "TOYOTA MOTOR ENGINEERING & MANUFACTURING", "GULF STATES TOYOTA INC"],
    "HONDA": ["AMERICAN HONDA MOTOR CO", "HONDA AMERICAN HONDA MOTOR CO"],
    "HYUNDAI": ["HYUNDAI MOTOR AMERICA", "HYUNDAI MOTOR COMPANY"],
    "CHRYSLER": ["DAIMLERCHRYSLER CORPORATION", "DAIMLERCHRYSLER MANUFACTUR INTERNATIONAL",
                 "DAIMLERCHRYSLER MANUFACTURING INTL", "DAIMLERCHRYSLER COMMERCIAL BUSES NC",
                 "DAIMLERCHRYSLER COMMERCIAL BUSES NA"],
    "TESLA": ["TESLA INC"],
    "ALFA ROMEO": ["ALFA ROMEO INCORPORATED"],
}
alias_lookup = {clean_name(v): k for k, variants in manual_aliases.items() for v in variants}

# --- Matching function: alias lookup first, fuzzy fallback second ---
def match_to_oem(raw_name, reference_list, threshold=90):
    if pd.isna(raw_name):
        return None
    cleaned = clean_name(raw_name)
    if cleaned in alias_lookup:
        return alias_lookup[cleaned]
    result = process.extractOne(cleaned, reference_list, scorer=fuzz.token_set_ratio)
    if result and result[1] >= threshold:
        return result[0]
    return None

# --- Rebuild lookup maps from scratch ---
complaint_names = complaints_df["MFR_NAME"].dropna().unique()
recall_names = recalls_df["MFGNAME"].dropna().unique()

complaint_map = {name: match_to_oem(name, oem_reference) for name in complaint_names}
recall_map = {name: match_to_oem(name, oem_reference) for name in recall_names}

complaints_df["OEM_CLEAN"] = complaints_df["MFR_NAME"].map(complaint_map)
recalls_df["OEM_CLEAN"] = recalls_df["MFGNAME"].map(recall_map)

print("Complaints matched:", complaints_df["OEM_CLEAN"].notna().sum(), "/", len(complaints_df))
print("Recalls matched:", recalls_df["OEM_CLEAN"].notna().sum(), "/", len(recalls_df))



Complaints matched: 2110341 / 2234970
Recalls matched: 150478 / 326100


In [19]:
# Keep only rows that matched a real passenger-vehicle OEM
complaints_clean = complaints_df[complaints_df["OEM_CLEAN"].notna()].copy()
recalls_clean = recalls_df[recalls_df["OEM_CLEAN"].notna()].copy()

print("Complaints kept:", len(complaints_clean), "/", len(complaints_df),
      f"({len(complaints_clean)/len(complaints_df):.1%})")
print("Recalls kept:", len(recalls_clean), "/", len(recalls_df),
      f"({len(recalls_clean)/len(recalls_df):.1%})")

# Sanity check: how many unique OEMs actually survived the filter?
print("\nUnique OEMs in complaints_clean:", complaints_clean["OEM_CLEAN"].nunique())
print("Unique OEMs in recalls_clean:", recalls_clean["OEM_CLEAN"].nunique())

# Quick look at volume by OEM — sniff-test that Ford/Toyota/GM dominate as expected
print("\nTop 15 OEMs by complaint volume:")
print(complaints_clean["OEM_CLEAN"].value_counts().head(15))

print("\nTop 15 OEMs by complaint volume:")
print(recalls_clean["OEM_CLEAN"].value_counts().head(15))

Complaints kept: 2110341 / 2234970 (94.4%)
Recalls kept: 150478 / 326100 (46.1%)

Unique OEMs in complaints_clean: 38
Unique OEMs in recalls_clean: 39

Top 15 OEMs by complaint volume:
OEM_CLEAN
GENERAL MOTORS    445082
FORD              431588
CHRYSLER          394202
TOYOTA            157638
HONDA             141814
NISSAN            107027
HYUNDAI            78739
VOLKSWAGEN         65710
KIA                61700
BMW                47263
SUBARU             34857
MERCEDES-BENZ      32563
MAZDA              29800
TESLA              22881
VOLVO              15232
Name: count, dtype: int64

Top 15 OEMs by complaint volume:
OEM_CLEAN
MERCEDES-BENZ     47072
FORD              17797
TOYOTA            15600
HONDA             12867
GENERAL MOTORS    10682
CHRYSLER           8029
VOLKSWAGEN         4853
NISSAN             4691
BMW                4480
LAND ROVER         3726
VOLVO              3604
PORSCHE            2564
HYUNDAI            2382
MITSUBISHI         2334
SUBARU             1679


## 5. Component hierarchy parsing & crosswalk

Both complaints and recalls describe the defective part using a shared  
colon-delimited hierarchy (e.g. `AIR BAGS:FRONTAL:SENSOR`). Rather than  
fuzzy-matching hundreds of detailed raw strings, the top-level category  
(text before the first colon) is extracted directly — this collapses both  
tables to a shared, comparable component vocabulary.

In [20]:
# Examine raw complaint component labels before standardization. The output shows
# whether the source contains a consistent hierarchy that can be parsed.
print(complaints_clean["COMPDESC"].value_counts())


COMPDESC
ELECTRICAL SYSTEM                                                      186868
ENGINE                                                                 165890
POWER TRAIN                                                            134328
AIR BAGS                                                               128294
STEERING                                                               116124
                                                                        ...  
EQUIPMENT ADAPTIVE/MOBILITY:WHEELCHAIR LIFT/RAMP:CRITICAL FASTENERS         1
EQUIPMENT:APPLIANCE:REFRIGERATOR                                            1
ELECTRICAL SYSTEM:PROPULSION SYSTEM:DC/DC CONVERTER                         1
EQUIPMENT:ELECTRICAL:ACCESSORY BATTERY:CHARGE MANAGEMENT(BMS)               1
EQUIPMENT:APPLIANCE:OVEN/STOVE/COOKTOP                                      1
Name: count, Length: 730, dtype: int64


In [21]:
print(recalls_clean["COMPNAME"].value_counts())

COMPNAME
ELECTRICAL SYSTEM:SOFTWARE                                                                                18431
COMMUNICATION: AUTO CRASH NOTIFICATION                                                                    16553
FUEL SYSTEM, GASOLINE:DELIVERY:FUEL PUMP                                                                   8356
BACK OVER PREVENTION:DISPLAY FUNCTION                                                                      7741
AIR BAGS:FRONTAL:PASSENGER SIDE:INFLATOR MODULE                                                            4330
                                                                                                          ...  
EQUIPMENT:ELECTRICAL:INFOTAINMENT                                                                             1
EQUIPMENT:MOTORCYCLE:LUGGAGE/SIDE/TOP CASES:MOUNTING HARDWARE/FASTENERS                                       1
ELECTRICAL SYSTEM:PROPULSION SYSTEM:SHOCK PROTECTION                                           

In [22]:
# Both COMPDESC (complaints) and COMPNAME (recalls) turned out to use a shared colon-delimited hierarchy 
# (e.g. "AIR BAGS:FRONTAL:SENSOR"), so instead of fuzzy-matching 730 and 539 raw strings, the top-level category 
# (text before the first colon) was extracted directly — a parsing problem, not a matching problem.

complaints_clean["COMP_TOP"] = complaints_clean["COMPDESC"].str.split(":").str[0].str.strip()
recalls_clean["COMP_TOP"] = recalls_clean["COMPNAME"].str.split(":").str[0].str.strip()

In [23]:
print("Complaints top-level categories:", complaints_clean["COMP_TOP"].nunique())
print("Recalls top-level categories:", recalls_clean["COMP_TOP"].nunique())

print(complaints_clean["COMP_TOP"].value_counts().head(30))
print(recalls_clean["COMP_TOP"].value_counts().head(30))

Complaints top-level categories: 53
Recalls top-level categories: 40
COMP_TOP
ELECTRICAL SYSTEM                     242275
POWER TRAIN                           213640
ENGINE                                165890
AIR BAGS                              155444
STEERING                              141955
ENGINE AND ENGINE COOLING             116864
UNKNOWN OR OTHER                      112196
SERVICE BRAKES, HYDRAULIC             103952
STRUCTURE                              98993
SUSPENSION                             86136
VEHICLE SPEED CONTROL                  83570
SERVICE BRAKES                         75966
EXTERIOR LIGHTING                      61048
FUEL/PROPULSION SYSTEM                 60627
FUEL SYSTEM, GASOLINE                  54177
VISIBILITY                             51816
SEAT BELTS                             38465
TIRES                                  37311
SEATS                                  28297
FORWARD COLLISION AVOIDANCE            27278
VISIBILITY/WIPER      

In [24]:
comp_set = set(complaints_clean["COMP_TOP"].unique())
recall_set = set(recalls_clean["COMP_TOP"].unique())

print("In both:", len(comp_set & recall_set))
print("Only in complaints:", comp_set - recall_set)
print("Only in recalls:", recall_set - comp_set)

In both: 40
Only in complaints: {nan, 'COMMUNICATIONS', 'OTHER', 'Insert, Padding', 'ROLLOVER', 'SERVICE BRAKES, HYDRAULIC; AUTOHOLD BRAKE SYSTEM/BRAKE HOLD', 'Other/I am not sure', 'I suspect the car seat is counterfeit', 'Chest Clip, Buckle, Harness', 'NONE', 'Carry Handle, Shell, Base', 'Tether, Lower Anchor (on car seat or vehicle)', 'Other/Unknown', 'FIRERELATED'}
Only in recalls: set()


### Handling leftover category mismatches

A few categories exist only on the complaints side — some are true naming  
variants (aliased), a few are legacy categories with no recall equivalent  
(kept as-is), and the rest are free-text/junk from car-seat-specific fields  
(measured and dropped only after confirming they're negligible).

In [25]:
# Check size of the junk/free-text bucket before deciding drop vs. map —
# don't build special-case handling for something that's negligible
junk = ['NONE', 'Insert, Padding', 'Tether, Lower Anchor (on car seat or vehicle)', 
        'Other/Unknown', float('nan'), 'Chest Clip, Buckle, Harness', 
        'I suspect the car seat is counterfeit', 'Other/I am not sure', 
        'Carry Handle, Shell, Base']

junk_count = complaints_clean["COMP_TOP"].isin(junk).sum() + complaints_clean["COMP_TOP"].isna().sum()
print("Junk/free-text rows:", junk_count, f"({junk_count/len(complaints_clean):.2%})")

Junk/free-text rows: 694 (0.03%)


In [26]:
# Apply the two alias fixes, then drop the negligible junk rows
alias_map = {
    "COMMUNICATIONS": "COMMUNICATION",
    "SERVICE BRAKES, HYDRAULIC; AUTOHOLD BRAKE SYSTEM/BRAKE HOLD": "SERVICE BRAKES, HYDRAULIC"
}
complaints_clean["COMP_TOP"] = complaints_clean["COMP_TOP"].replace(alias_map)

# Drop junk rows now that we've confirmed they're negligible
complaints_clean = complaints_clean[~complaints_clean["COMP_TOP"].isin(junk)].copy()
complaints_clean = complaints_clean[complaints_clean["COMP_TOP"].notna()].copy()

In [27]:
# Final alias map for COMPONENT_CLEAN — only collapsing true duplicates,
# keeping brake/fuel subtypes split since they're mechanically distinct
component_alias_map = {
    "ENGINE AND ENGINE COOLING": "ENGINE",
    "VISIBILITY/WIPER": "VISIBILITY"
}

complaints_clean["COMPONENT_CLEAN"] = complaints_clean["COMP_TOP"].replace(component_alias_map)
recalls_clean["COMPONENT_CLEAN"] = recalls_clean["COMP_TOP"].replace(component_alias_map)

# Sanity check — category count should drop slightly, not collapse everything
print("Complaints categories after collapse:", complaints_clean["COMPONENT_CLEAN"].nunique())
print("Recalls categories after collapse:", recalls_clean["COMPONENT_CLEAN"].nunique())

Complaints categories after collapse: 41
Recalls categories after collapse: 38


In [28]:
# Drop intermediate parsing column — COMPONENT_CLEAN supersedes it,
# no reason to carry both into SQL Server
complaints_clean = complaints_clean.drop(columns=["COMP_TOP"])
recalls_clean = recalls_clean.drop(columns=["COMP_TOP"])

## 6. Final validation

Confirms both cleaned tables now share a common `COMPONENT_CLEAN` vocabulary,  
which is required before the manufacturer+component join happens in SQL Server.

In [29]:
comp_set = set(complaints_clean["COMPONENT_CLEAN"].unique())
recall_set = set(recalls_clean["COMPONENT_CLEAN"].unique())

print("In both:", len(comp_set & recall_set))
print("Only in complaints:", comp_set - recall_set)
print("Only in recalls:", recall_set - comp_set)

In both: 38
Only in complaints: {'OTHER', 'ROLLOVER', 'FIRERELATED'}
Only in recalls: set()


In [30]:
# Renaming raw/messy source columns to _RAW and their standardized
# counterparts to _CLEAN, plus fixing NHTSA's non-descriptive TXT suffixes —
# same renames you locked in and applied via sp_rename in SQL Server

# complaints_clean
complaints_clean = complaints_clean.rename(columns={
    "MFR_NAME": "MFR_NAME_RAW",
    "OEM_CLEAN": "MFR_NAME_CLEAN",
    "YEARTXT": "MODEL_YEAR",
    "COMPDESC": "COMPONENT_RAW",
    "CDESCR": "ACTUAL_COMPLAINT_TEXT",
    "MAKETXT": "VEHICLE_MAKE",
    # add this one only if you decided to rename MODELTXT too
    "MODELTXT": "VEHICLE_MODEL",
})

# recalls_clean
recalls_clean = recalls_clean.rename(columns={
    "MFGNAME": "MFR_NAME_RAW",
    "OEM_CLEAN": "MFR_NAME_CLEAN",
    "YEARTXT": "MODEL_YEAR",
    "COMPNAME": "COMPONENT_RAW",
    "MAKETXT": "VEHICLE_MAKE",
    "MODELTXT": "VEHICLE_MODEL",
    # fixing NHTSA's misspelling while you're in here — drop this line
    # if you'd rather keep the column name matching the source exactly
    "CONEQUENCE_DEFECT": "CONSEQUENCE_DEFECT",
})

# sanity check — confirm no leftover old names, no accidental duplicates
print(complaints_clean.columns.tolist())
print(recalls_clean.columns.tolist())

['CMPLID', 'ODINO', 'MFR_NAME_RAW', 'VEHICLE_MAKE', 'VEHICLE_MODEL', 'MODEL_YEAR', 'COMPONENT_RAW', 'DATEA', 'CRASH', 'FIRE', 'INJURED', 'DEATHS', 'ACTUAL_COMPLAINT_TEXT', 'MFR_NAME_CLEAN', 'COMPONENT_CLEAN']
['CAMPNO', 'MFR_NAME_RAW', 'VEHICLE_MAKE', 'VEHICLE_MODEL', 'MODEL_YEAR', 'COMPONENT_RAW', 'RCDATE', 'DESC_DEFECT', 'CONSEQUENCE_DEFECT', 'MFR_NAME_CLEAN', 'COMPONENT_CLEAN']


In [31]:
complaints_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2109782 entries, 0 to 2234969
Data columns (total 15 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   CMPLID                 int64         
 1   ODINO                  int64         
 2   MFR_NAME_RAW           object        
 3   VEHICLE_MAKE           object        
 4   VEHICLE_MODEL          object        
 5   MODEL_YEAR             float64       
 6   COMPONENT_RAW          object        
 7   DATEA                  datetime64[ns]
 8   CRASH                  object        
 9   FIRE                   object        
 10  INJURED                int64         
 11  DEATHS                 int64         
 12  ACTUAL_COMPLAINT_TEXT  object        
 13  MFR_NAME_CLEAN         object        
 14  COMPONENT_CLEAN        object        
dtypes: datetime64[ns](1), float64(1), int64(4), object(9)
memory usage: 257.5+ MB


In [32]:
complaints_clean.loc[
    complaints_clean['MODEL_YEAR'] == 9999,
    'MODEL_YEAR'
] = pd.NA

recalls_clean.loc[
    recalls_clean['MODEL_YEAR'] == 9999,
    'MODEL_YEAR'
] = pd.NA

In [33]:
# # Push cleaned dataframes into SQL Server as tables — from here on,
# # SQL handles aggregation/window functions, not pandas
# import sqlalchemy

# engine = sqlalchemy.create_engine(
#     "mssql+pyodbc://localhost/NHTSA?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes",
#     fast_executemany=True   # batches inserts instead of row-by-row — much faster for large tables
# )

# complaints_clean.to_sql("complaints_clean", engine, if_exists="replace", index=False, chunksize=10000)
# recalls_clean.to_sql("recalls_clean", engine, if_exists="replace", index=False, chunksize=10000)